In [ ]:
import pandas as pd
import nflreadpy as nfl

# 1. Download 2023 weekly player stats
print("Downloading weekly stats...")
weekly_stats = nfl.load_player_stats([2023]).to_pandas()

# 2. Download 2023 schedule & closing lines
print("Downloading schedules and closing lines...")
schedules = nfl.load_schedules([2023]).to_pandas()

# 3. Standardize column data types to prevent merge mismatches
weekly_stats['season'] = weekly_stats['season'].astype(int)
weekly_stats['week'] = weekly_stats['week'].astype(int)
weekly_stats['team'] = weekly_stats['team'].astype(str).str.strip()

schedules['season'] = schedules['season'].astype(int)
schedules['week'] = schedules['week'].astype(int)

# 4. Melt schedule and name the value 'team' to match weekly_stats
schedule_melted = pd.melt(
    schedules, 
    id_vars=['game_id', 'season', 'week', 'spread_line', 'total_line'], 
    value_vars=['home_team', 'away_team'],
    var_name='home_away', 
    value_name='team'  # Changed from 'team_abbr' to 'team'
)
schedule_melted['team'] = schedule_melted['team'].astype(str).str.strip()

# 5. Merge on standardized columns including game_id
print("Merging data pipelines...")
merged_data = pd.merge(
    weekly_stats, 
    schedule_melted, 
    on=['game_id', 'season', 'week', 'team'],  # Includes game_id to avoid suffixes
    how='inner'  # Inner guarantees only records with matching schedules show up
)

print(f"Merge successful! Total records: {merged_data.shape[0]}")

# 6. Verify with columns guaranteed to be in the dataframe
cols_to_view = ['game_id', 'player_display_name', 'team', 'week', 'passing_yards', 'spread_line']
print(merged_data[cols_to_view].head())

Merging data pipelines...
Merge successful! Total records: 18643


KeyError: "['game_id'] not in index"